# 11 · Evaluation on the held-out women

Everything here is measured on the 1,375 held-out women, with their survey weights, and every
interval comes from resampling **whole clusters within their strata** (500 times), so it
reflects the survey's design.

For each model, predictor set and rung: average precision against chance (the share reaching
the rung), AUROC, the Brier score, calibration (slope and intercept), the share of the women one
is looking for who sit in the top fifth of the ranking, sensitivity and specificity at the
threshold chosen on the development women, AUROC within regions, wealth, education and
residence groups, the difference from the best model and from logistic regression with its
paired interval, and the net benefit across risk thresholds (decision curve).

Two further checks: every model is refitted from its saved settings and must reproduce its
saved predictions exactly, and the tuning is examined: what the search gained over the model's
default settings, and which settings mattered.

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

MODELS = ["Logistic regression", "Random Forest", "XGBoost"]

In [ ]:
repro.run_script("fit_library_models.py", "--root", P.results, "--lineup", "three")
repro.run_script("run_hyperparameter_ablation.py", "--root", P.results, "--models", *MODELS)
repro.run_script("run_explainability.py")   # the cards show what each model leans on (notebook 12)
repro.run_script("run_model_cards.py", "--library", P.results, "--lineup", "three", *(["--quick"] if QUICK else []))

## How many predictors: the four sets against each other

Held-out average precision of each model on each set. The study's finding: for ANC1+ and ANC4+
the full pool is best; for ANC8+ the short set does as well as the pool and the 21.

In [ ]:
cards = json.load(open(P.outputs / "model_cards" / "model_cards.json"))
rows = []
for key, by in cards["cards"].items():
    fam = cards["families"][key]
    for rung, c in by.items():
        rows.append({"set": lib.SET_LABEL[fam["set"]], "model": fam["name"], "rung": rung, "AP": c["ap"],
                     "AP 95%": f"{c['ap_ci'][0]:.3f}–{c['ap_ci'][1]:.3f}", "AUROC": c["auroc"], "slope": c["slope"]})
cardtab = pd.DataFrame(rows)
base = cardtab[cardtab.model.isin(MODELS)]
print(base.pivot_table(index=["rung", "model"], columns="set", values="AP").round(3).to_string())

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=False)
for ax, rung in zip(axes, ["anc1", "anc4", "anc8"]):
    t = base[base.rung == rung].pivot_table(index="set", columns="model", values="AP")
    t.plot.bar(ax=ax, rot=0, width=0.75, legend=(rung == "anc8"))
    ax.set_title(rung.upper().replace("ANC", "ANC") + "+"); ax.set_xlabel(""); ax.set_ylim(t.min().min() - 0.02, t.max().max() + 0.01)
axes[0].set_ylabel("held-out average precision")
plt.tight_layout(); plt.show()

## Every model on the full pool, with intervals

In [ ]:
print(cardtab[cardtab.set == "Full pool"].sort_values(["rung", "AP"], ascending=[True, False])
      [["rung", "model", "AP", "AP 95%", "AUROC", "slope"]].round(3).to_string(index=False))

## What tuning bought

In [ ]:
for rung in lib.RUNGS:
    a = pd.read_csv(P.outputs / "library" / "tuning" / f"ablation_pool_{rung}.csv")
    g = a[a.parameter.isin(["(library defaults)", "(tuned optimum)"])].pivot_table(index="model", columns="parameter", values="AP")
    g["gain"] = g["(tuned optimum)"] - g["(library defaults)"]
    print(rung); print(g.round(4).to_string(), "\n")
repro.compare_cells(P.results, MODELS)
repro.report()